# 📈 Institutional Financial RAG & Multi-Agent System

This notebook allows you to test that the **Hierarchical Multi-Agent RAG** is running properly and execute an **interactive test chat**.

### Architecture Overview:
- **CIO Supervisor ReActAgent**: Chief Investment Officer orchestrating quantitative & qualitative research.
- **Fundamental Analyst (Sub-Agent)**: Deterministic quantitative queries against SQLite (`foundamental_sql.db`).
- **Market News Analyst (Sub-Agent)**: Semantic vector search + Cross-Encoder reranker over ChromaDB (`news_vector`).
- **Synthesis**: Qwen 2.5 Local LLM via Ollama.

In [ ]:
# Step 0: Ensure correct sys.path whether running from workspace root or rag/ directory
%load_ext autoreload
%autoreload 2

import os
import sys
import json
import time
import torch

# Purge cached rag modules so edits to .py files take effect without kernel restart
for mod in list(sys.modules.keys()):
    if mod.startswith('rag.') or mod == 'rag':
        del sys.modules[mod]

# Add workspace root to sys.path
nb_dir = os.path.abspath('')
workspace_dir = os.path.abspath('..') if os.path.basename(nb_dir) == 'rag' else nb_dir
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)

from config import CONFIG
from get_data import load_fundamentals
from rag import (
    FinancialRAGEngine,
    check_ollama_status,
    get_chroma_document_count,
    index_all_data,
)

print('✅ Packages and modules loaded successfully!')
print(f'CUDA Acceleration Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device: {torch.cuda.get_device_name(0)}')
else:
    print('Hardware: CPU mode')


## Step 1: Check System Health (Ollama, SQLite, ChromaDB)
Verifies local Ollama service, model installation, SQLite stock count, and ChromaDB vector documents.

In [ ]:
# 1. Check Ollama LLM service
status = check_ollama_status()
print("=== 🤖 Ollama Status ===")
print(json.dumps(status, indent=2))

if status["status"] != "online":
    print(f"\n⚠️ Note: Target model '{status['target_model']}' is not ready.")
    print(f"Start Ollama by running in terminal: ollama run {status['target_model']}")
else:
    print(f"\n✅ Target model '{status['target_model']}' is ONLINE!")

# 2. Check SQLite & ChromaDB counts
doc_count = get_chroma_document_count(CONFIG.chroma_dir, CONFIG.chroma_collection)
fundamentals = load_fundamentals()
print(f"\n=== 📊 Storage Health ===")
print(f"• SQLite Fundamentals Database: {len(fundamentals)} stocks indexed")
print(f"• ChromaDB News Vector Store: {doc_count} document chunks indexed")


## Step 2: (Optional) Synchronize ChromaDB Vector Store
If document count is 0 or you updated `data/news.json`, run this cell to sync the vector embeddings.

In [ ]:
if doc_count == 0:
    print("ChromaDB index is empty. Building index from news data...")
    index = index_all_data()
    print("✅ ChromaDB vector index built.")
else:
    print(f"✅ ChromaDB vector index already populated with {doc_count} documents. Ready to query!")


## Step 4: Initialize Financial RAG Engine
Initializes the singletons for the embedding model, cross-encoder reranker, Ollama LLM, and the vector index.

In [ ]:
print("Initializing FinancialRAGEngine...")
engine = FinancialRAGEngine()
print("✅ Financial RAG Engine ready!")


## Step 5: Execute Multi-Agent Financial Query (Native Async)
Queries the hierarchical multi-agent system using top-level `await engine.aquery()`. The CIO Supervisor dynamically evaluates your intent and delegates to domain sub-agents.

In [ ]:
# Test query that combines valuation and recent catalysts (Native Async)
test_query = "What is google's current valuation (P/E, ROIC) and what are the main recent catalysts?"

print(f"Executing Multi-Agent Query (Native Async): '{test_query}'\n")
result = await engine.aquery(query_text=test_query)

print("\n=================================================================")
print("🧠 CIO EXECUTIVE SYNTHESIS")
print("=================================================================\n")
print(result.get("answer") or result.get("response"))

print("\n-----------------------------------------------------------------")
print(f"⏱️ Latency: {result.get('latency_seconds', 0)}s | Model: {result.get('model')} | Agent: {result.get('agent')}")

trace = result.get("trace", {})
print(f"Supervisor: {trace.get('supervisor')}")
print(f"Sub-Agents Invoked: {trace.get('agents')}")
print(f"Total Delegation Calls: {trace.get('iterations')}")
print("-----------------------------------------------------------------")

print("\n=================================================================")
print("🔍 COMPLETE SUB-AGENT EXECUTION, VARIABLES & RAW DATA")
print("=================================================================")

sources = result.get("sources", [])
for i, src in enumerate(sources, 1):
    agent_name = src.get("agent", src.get("tool_name", "Sub-Agent"))
    tool_name = src.get("tool_name", "")
    tool_args = src.get("tool_kwargs", {})
    full_output = src.get("output") or src.get("output_preview", "")
    
    print(f"\n" + ("#" * 65))
    print(f"📌 [DELEGATION STEP {i}]: Supervisor Called '{tool_name}'")
    print(f"   • Sub-Agent: {agent_name}")
    print(f"   • Variables Passed from CIO: {json.dumps(tool_args, indent=5)}")
    print("#" * 65)
    print("\n--- [SUB-AGENT OUTPUT & RAW TOOL DATA] ---")
    print(full_output.strip())
    print("-" * 65)


## Step 6: Real-Time Event & Token Streaming (`astream_pipeline_events`)
Demonstrates native async live stage-by-stage pipeline progress tracking and word-by-word streaming generation directly via `async for`.

In [ ]:
stream_query = "Compare Microsoft (MSFT) and Apple (AAPL) gross margins and balance sheet strength."

print(f"🔎 Live Streaming Query (Native Async): '{stream_query}'\n")
print("--- Pipeline Execution Trace ---")

async for event in engine.astream_pipeline_events(stream_query):
    if event["type"] == "step":
        stage = event.get("stage", "").upper()
        title = event.get("title", "")
        msg = event.get("message", "")
        print(f"[{stage}] {title}: {msg}")
    elif event["type"] == "token":
        sys.stdout.write(event.get("content", ""))
        sys.stdout.flush()
    elif event["type"] == "done":
        meta = event.get("metadata", {})
        print(f"\n\n✅ Completed in {meta.get('latency_seconds')}s (Model: {meta.get('model')})")
        print(f"Analyst sources retrieved: {len(event.get('citations', []))}")


## Step 7: Institutional Stock Memo & Comparison Tools
Generate structured Bull/Bear memos and side-by-side comparative stock analyses with full verbose evidence.

In [ ]:
# 1. Generate Bull / Bear Investment Thesis for a ticker (Native Async)
ticker = "NVDA"
print(f"Generating institutional memo for {ticker}...")
thesis_result = await engine.agenerate_stock_thesis(ticker=ticker)

print(f"\n=================== 📊 Institutional Research Memo: {ticker} ===================\n")
print(thesis_result.get("thesis"))
print(f"\n⏱️ Latency: {thesis_result.get('latency_seconds')}s | Model: {thesis_result.get('model')}")

sources = thesis_result.get("citations", thesis_result.get("sources", []))
print(f"\nEvaluated Sub-Agent Sources ({len(sources)}):\n")
for i, s in enumerate(sources, 1):
    full_out = s.get('output') or s.get('output_preview', '')
    print(f"[{i}] {s.get('agent', s.get('tool_name'))}:\n")
    print(f"    Variables: {s.get('tool_kwargs', {})}\n")
    print("-" * 50)
    print(full_out.strip())
    print("-" * 50)


## Step 8: Interactive Test Chat Session (Native Async) 💬
Run this cell to start an interactive multi-turn chat session with the CIO Financial RAG Assistant.
- Enter any financial question (valuation, metrics, screeners, or news catalysts).
- Supports multi-turn memory buffer across questions.
- Type `exit`, `quit`, or `q` to stop.
- Type `clear` to reset the chat memory.

In [ ]:
async def run_test_chat():
    """
    Interactive test chat loop with the Financial RAG CIO Supervisor (Native Async).
    Queries the multi-agent system and displays answers with sub-agent sources.
    """
    print("=" * 65)
    print("💬 Financial RAG Interactive Test Chat (Native Async)")
    print("Chief Investment Officer (CIO) Supervisor Multi-Agent Assistant")
    print("Type your question below, or 'exit' / 'quit' to stop.")
    print("=" * 65)

    sample_queries = [
        "What is NVDA P/E ratio and ROIC?",
        "Find companies with ROIC > 20% and gross margin > 50%",
        "What are the latest news and catalysts for AAPL?",
        "Compare NVDA vs GOOGL valuation and profitability",
    ]
    print("\n💡 Suggested test prompts:")
    for sq in sample_queries:
        print(f"   • {sq}")
    print("-" * 65)

    session_id = "jupyter_chat_session"

    while True:
        try:
            user_input = input("\nInvestor: ").strip()
        except (KeyboardInterrupt, EOFError):
            print("\nSession ended.")
            break

        if not user_input:
            continue

        if user_input.lower() in ["exit", "quit", "q"]:
            print("Ending chat session. Goodbye!")
            break

        if user_input.lower() == "clear":
            engine.reset_session_memory(session_id)
            print("\n" * 2 + "--- Chat & Memory Cleared ---")
            continue

        print(f"\n[CIO Assistant is analyzing across SQLite & ChromaDB...]")
        start_t = time.time()
        try:
            res = await engine.aquery(user_input, session_id=session_id)
            ans = res.get("answer") or res.get("response", "No response generated.")
            latency = res.get("latency_seconds", round(time.time() - start_t, 2))
            
            print(f"\nCIO Verdict ({latency}s):\n")
            print(ans)

            sources = res.get("sources", [])
            if sources:
                print("\n📌 Analyst Sources & Evidence Consulted:")
                for i, s in enumerate(sources, 1):
                    agent_name = s.get("agent", s.get("tool_name", "Analyst"))
                    full_out = s.get("output") or s.get("output_preview", "")
                    print(f"\n   [{i}] {agent_name} (tool: {s.get('tool_name')} | args: {s.get('tool_kwargs', {})}):\n   {full_out.strip()}")
            print("-" * 65)
        except Exception as e:
            print(f"\n❌ Error during query: {e}")
            print("-" * 65)

# Launch interactive chat
await run_test_chat()


## Step 9: Quick Single-Turn Test Chat Function (Native Async)
If you want to run quick test questions in individual notebook cells without using the interactive `input()` loop, use `await quick_chat(question)`:

In [ ]:
async def quick_chat(question: str, session_id: str = "quick_chat"):
    """Runs a single question through the RAG engine and prints the answer and citations (Native Async)."""
    print(f"🧑‍💻 Investor: {question}\n")
    res = await engine.aquery(question, session_id=session_id)
    print("🤖 CIO Assistant:\n")
    print(res.get("answer") or res.get("response"))
    print(f"\n⏱️ Latency: {res.get('latency_seconds', 0)}s")
    sources = res.get("sources", [])
    if sources:
        print("\n📚 Complete Analyst Sources:")
        for i, s in enumerate(sources, 1):
            full_out = s.get('output') or s.get('output_preview', '')
            print(f"\n  [{i}] {s.get('agent', s.get('tool_name'))} (args: {s.get('tool_kwargs', {})}):\n  {full_out.strip()}")

# Example single-turn chat
await quick_chat("What are the key financial strengths and risks of NVDA?")
